In [0]:
from pyspark.sql import functions as F

raw_df = spark.read.parquet("s3://ledgr-raw-data-2026/processed/")

print(f"Rows read from S3: {raw_df.count()}")
print(f"Columns: {raw_df.columns}")

In [0]:
from delta.tables import DeltaTable

bronze_table = "ledgr.bronze.calls_raw"

if spark.catalog.tableExists(bronze_table):
    # Table exists: MERGE to dedupe on attempt_id, avoiding double-counting on reruns
    delta_table = DeltaTable.forName(spark,bronze_table)
    (delta_table.alias("target")
    .merge(raw_df.alias("source"), "target.attempt_id = source.attempt_id")
    .whenNotMatchedInsertAll()
     .execute())
    print("Merged into existing Bronze table (idempotent, no duplicates)")
else:
     # First run: create the table
     raw_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)
     print(f"Created new Bronze table: {bronze_table}")
# Verify
result_count = spark.table(bronze_table).count()
print(f"Bronze table row count: {result_count}")

In [0]:
# Idempotency test: rerun the same read+merge logic and confirm no duplication
raw_df_rerun = spark.read.parquet("s3://ledgr-raw-data-2026/processed/")

from delta.tables import DeltaTable
delta_table = DeltaTable.forName(spark,bronze_table)
(delta_table.alias("target")
.merge(raw_df_rerun.alias("source"), "target.attempt_id = source.attempt_id")
.whenNotMatchedInsertAll()
.execute())

recount = spark.table("ledgr.bronze.calls_raw").count()
print(f"Row count after rerun: {recount}")
print(f"Expected (no duplication): 265091")
print(f"MATCH: {recount == 265091}")

In [0]:
# Drop the incorrectly-scoped Bronze table from earlier
spark.sql("DROP TABLE IF EXISTS ledgr.bronze.calls_raw")
print("Dropped incorrect Bronze table")

# Read the TRUE raw dataset — untouched, nested spans column, one row per session
true_raw_df = spark.read.parquet("s3://ledgr-raw-data-2026/raw/")
print(f"Raw rows read: {true_raw_df.count()}")
print(f"Columns: {true_raw_df.columns}")
true_raw_df.printSchema()

In [0]:
bronze_table = "ledgr.bronze.sessions_raw"

true_raw_df.write.format("delta").mode("overwrite").saveAsTable(bronze_table)
print(f"Created Bronze table: {bronze_table}")

result_count = spark.table(bronze_table).count()
print(f"Bronze table row count: {result_count}")

In [0]:
%pip install pytest

In [0]:
import subprocess
result = subprocess.run(["pytest", "/Workspace/Users/sreelakshmitd97@gmail.com/ledgr/databricks/tests/", "-v"], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

In [0]:
import subprocess
import os

env = os.environ.copy()
env["PYTHONDONTWRITEBYTECODE"] = "1"

result = subprocess.run(
    ["pytest", "-p", "no:cacheprovider", "--assert=plain",
     "/Workspace/Users/sreelakshmitd97@gmail.com/ledgr/databricks/tests/", "-v"],
    capture_output=True, text=True, env=env
)
print(result.stdout)
print(result.stderr)

In [0]:
import subprocess
import os

env = os.environ.copy()
env["PYTHONDONTWRITEBYTECODE"] = "1"
env["PYTHONPATH"] = "/Workspace/Users/sreelakshmitd97@gmail.com/ledgr:" + env.get("PYTHONPATH", "")

result = subprocess.run(
    ["pytest", "-p", "no:cacheprovider", "--assert=plain",
     "/Workspace/Users/sreelakshmitd97@gmail.com/ledgr/ledgr_databricks/tests/", "-v"],
    capture_output=True, text=True, env=env
)
print(result.stdout)
print(result.stderr)